# Preparación de datos

En esta notebook se construye el dataset final para modelado a partir de los datos crudos.

Se aplican transformaciones orientadas a capturar patrones temporales y de negocio:

- Estandarización de fechas y consistencia temporal
- Creación de features de estacionalidad
- Incorporación de variables de calendario
- Generación de lags para modelar dependencia temporal en las ventas

El dataset resultante queda preparado para ser utilizado en el entrenamiento de modelos predictivos.

### Imports

In [ ]:
# Core
from pyspark.sql import SparkSession

# Utils
import os
from pyspark.sql.functions import to_date, year, month, dayofweek, weekofyear, when, min, max, col

# Time series
from pyspark.sql.window import Window
from pyspark.sql.functions import lag

Sesion de spark

In [ ]:
spark = SparkSession.builder \
    .appName("penguin-data-prep") \
    .getOrCreate()


## Lectura de datos

In [3]:
path_actual = os.getcwd()
path_raw_ds_rs = os.path.join(path_actual, "..", "data", "raw", "dataset_ds_rs.csv")


In [4]:
df = spark.read.csv(path_raw_ds_rs, header=True, inferSchema=True)

df.show(5)
df.printSchema()

+-------+----------+---------------+------------+---------------+-----+-------+
|country|     fecha|marketing_spend|discount_pct|stock_available|price|  sales|
+-------+----------+---------------+------------+---------------+-----+-------+
|     AR|01/01/2023|           2360|       0.055|            321|46.14|2031.51|
|     AR|01/01/2023|           5926|       0.138|            622|46.25|4387.11|
|     AR|01/02/2023|           1630|       0.006|           1019|51.93|2769.57|
|     AR|01/02/2023|           4058|       0.042|            725|63.77|3249.06|
|     AR|01/03/2023|           3306|       0.137|            936|47.06|3881.05|
+-------+----------+---------------+------------+---------------+-----+-------+
only showing top 5 rows
root
 |-- country: string (nullable = true)
 |-- fecha: string (nullable = true)
 |-- marketing_spend: integer (nullable = true)
 |-- discount_pct: double (nullable = true)
 |-- stock_available: integer (nullable = true)
 |-- price: double (nullable = true

In [5]:
df.dtypes

[('country', 'string'),
 ('fecha', 'string'),
 ('marketing_spend', 'int'),
 ('discount_pct', 'double'),
 ('stock_available', 'int'),
 ('price', 'double'),
 ('sales', 'double')]

In [6]:
df.select("country").distinct().show()

+-------+
|country|
+-------+
|     CL|
|     UY|
|     AR|
+-------+



In [7]:
df.select(
    min("fecha").alias("min_fecha"),
    max("fecha").alias("max_fecha")
).show()

+----------+----------+
| min_fecha| max_fecha|
+----------+----------+
|01/01/2023|01/12/2025|
+----------+----------+



## Parseo de fecha

Las fechas se analizan utilizando el formato original DD/MM/YYYY y se almacenan como tipo de fecha, lo que permite operaciones temporales.

In [8]:
df = df.withColumn(
    "fecha_parsed",
    to_date("fecha", "dd/MM/yyyy")
)

df.select("fecha", "fecha_parsed").show(5)

+----------+------------+
|     fecha|fecha_parsed|
+----------+------------+
|01/01/2023|  2023-01-01|
|01/01/2023|  2023-01-01|
|01/02/2023|  2023-02-01|
|01/02/2023|  2023-02-01|
|01/03/2023|  2023-03-01|
+----------+------------+
only showing top 5 rows


## Features de fecha

Features útiles para detectar patrones de consumo.

Año, mes, dia de la semana (1 es para domingo)

In [9]:
df = df.withColumn("year", year("fecha_parsed")) \
       .withColumn("month", month("fecha_parsed")) \
       .withColumn("day_of_week", dayofweek("fecha_parsed"))

Fines de semana

In [10]:
df = df.withColumn(
    "is_weekend",
    when(df.day_of_week.isin([1, 7]), 1).otherwise(0)
)

Flag dias no laborables

In [11]:
holidays = [
    "2023-01-01",  # Año Nuevo
    "2023-03-24",  # Memoria
    "2023-04-02",  # Malvinas
    "2023-05-01",  # Trabajo
    "2023-05-25",  # Revolución de Mayo
    "2023-07-09",  # Independencia
    "2023-12-25"   # Navidad
]
from pyspark.sql.functions import col

df = df.withColumn(
    "is_holiday",
    col("fecha_parsed").isin(holidays).cast("int")
)

df = df.withColumn(
    "is_non_labour",
    (col("is_weekend") + col("is_holiday") > 0).cast("int")
)

## Features de tiempo

In [12]:
window_spec = Window.partitionBy("country").orderBy("fecha_parsed")

lags del dia anterior y la semana anterior

In [13]:
df = df.withColumn("lag_1", lag("sales", 1).over(window_spec)) \
       .withColumn("lag_7", lag("sales", 7).over(window_spec))

df = df.dropna(subset=["lag_1", "lag_7"])

## Guardado datos

In [ ]:
path_processed = os.path.join(path_actual, "..", "data", "processed", "data.parquet")
df.write.parquet(path_processed)